# **PetVision_v2**

RAG-based vet clinic internal assistant agentic chatbot for veterinarians that generates responses based on internal databases.

* 3 Classes: Demodicosis, Dermatitis, Ringworm
* 1 Pet: Dog


# **Installation and Setup**

In [ ]:
# Install LangChain ecosystem
!pip install -U --no-cache-dir \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
rank-bm25 \
pypdf \
bs4 \
requests

# Install supabase
!pip install supabase psycopg2-binary sqlalchemy

In [ ]:
# Install secret dependecies
from google.colab import userdata

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup LangSmith
!export LANGSMITH_TRACING="true"
!export LANGSMITH_API_KEY=userdata.get('LANGSMITH_API_KEY')

# Setup chat model
# Qwen2.5-1.5B: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

# Setup embedding model
# Embedding Model: Bge-small-en-v1.5 https://huggingface.co/BAAI/bge-small-en-v1.5
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

# Setup vector store
# Vector Store: In-Memory (Diagnosis / Treatment Papers)
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embedding=embeddings)

# Setup supabase
# PostgreSQL: Pets / Visits / Medical Records
from sqlalchemy import create_engine, text
from google.colab import userdata

DB_URL = userdata.get("SUPABASE_DB_URL")
engine = create_engine(DB_URL)

# Setup NER model
# NER: distilbert-base-uncased-finetuned-conll03-english
# https://huggingface.co/elastic/distilbert-base-uncased-finetuned-conll03-english
from transformers import pipeline as hf_pipeline

ner_model = hf_pipeline(
    "ner",
    model="elastic/distilbert-base-uncased-finetuned-conll03-english",
    aggregation_strategy="simple"  # merges subword tokens into full words
)

# Declare global variable for reference list
_last_references = []

# **Load PDFs and Metadata from Google Drive**

In [ ]:
# Load PDFs
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

# Define variables
PDF_FOLDER_PATH = "/content/drive/MyDrive/PetVision_v2/data/pdfs"
docs = []

# Loop through every file inside the PDF folder
for file in os.listdir(PDF_FOLDER_PATH):

  if file.endswith(".pdf"):
    # Create the full file path
    path = os.path.join(PDF_FOLDER_PATH, file)

    # Initialize a loader for the PDF
    loader = PyPDFLoader(path)

    # Load all content from the PDF
    pdf_docs = loader.load()

    # Attach the PDF file name as metadata to each document page
    for d in pdf_docs:
        d.metadata["file_name"] = file

    docs.extend(pdf_docs)

# 1 page in PDF = 1 doc
print("Loaded docs:", len(docs))

In [ ]:
# Load metadata.csv
import pandas as pd

# Define path
METADATA_FILE_PATH = "/content/drive/MyDrive/PetVision_v2/data/metadata.csv"

# Load the CSV file into a pandas DataFrame
metadata_df = pd.read_csv(METADATA_FILE_PATH)

# Convert the metadata table into a dictionary
# Key   = file_name
# Value = all metadata related to that file
metadata_map = metadata_df.set_index("file_name").to_dict(orient="index")

# Attach CSV metadata to each loaded PDF document
for d in docs:

  # Get the file name from the document metadata
  file_name = d.metadata["file_name"]

  # Check if the file exists inside metadata.csv
  if file_name in metadata_map:

      # Get the metadata row corresponding to this file
      row = metadata_map[file_name]

      # Add each metadata field into the document metadata
      d.metadata["doc_id"] = row["doc_id"]
      d.metadata["title"] = row["title"]
      d.metadata["disease"] = row["disease"]
      d.metadata["species"] = row["species"]
      d.metadata["document_type"] = row["document_type"]
      d.metadata["focus_area"] = row["focus_area"]
      d.metadata["source_journal"] = row["source"]
      d.metadata["year"] = row["year"]
      d.metadata["language"] = row["language"]
      d.metadata["url"] = row["url"]


# **Router**

## Semantic Routing

Routes to PostgreSQL / Vector Store (Embedding-based).

In [ ]:
import numpy as np

# Initialize embedding model wrapper
router_embedder = HuggingFaceEmbeddings(
  model_name="BAAI/bge-small-en-v1.5",
  encode_kwargs={"normalize_embeddings": True},
)

# Define keyword-rich semantic profiles for each disease
disease_profiles = {
  "Ringworm": """
  ringworm dermatophytosis fungal infection dermatophyte
  Microsporum Trichophyton circular bald patches scaly ring
  contagious zoonotic cats dogs humans
  antifungal treatment griseofulvin itraconazole miconazole
  """,

  "Dermatitis": """
  dermatitis atopic dermatitis canine atopic dermatitis
  allergic skin disease pruritus itching scratching
  hypersensitivity environmental allergens pollen dust mites
  flea allergy contact dermatitis seborrheic dermatitis
  oclacitinib apoquel cytopoint dupilumab corticosteroids
  immunotherapy allergen-specific IgE skin barrier
  """,

  "Demodicosis": """
  demodicosis Demodex canis mite infestation
  follicular mange generalized localized alopecia
  immunosuppressed dogs pustules crusting
  afoxolaner fluralaner ivermectin amitraz treatment
  """
}

# Precompute embeddings for each disease profile
disease_embeddings = {

    # Convert disease description into dense vector
    d: router_embedder.embed_query(text)
    for d, text in disease_profiles.items()
}

In [ ]:
# Route diseases
def route_diseases(query, top_k=1, threshold=0.25):

  # Convert incoming user query into embedding vector
  q_emb = np.array(router_embedder.embed_query(query))

  # Compute similarity score between query vector and each disease vector
  scores = {
    d: np.dot(q_emb, np.array(emb))
    for d, emb in disease_embeddings.items()
  }

  # Rank diseases by similarity score
  ranked = sorted(scores, key=scores.get, reverse=True)

  # Get best score for confidence check
  top_score = scores[ranked[0]]

  # If confidence is too low, fall back to returning all diseases
  if top_score < threshold:
    return list(scores.keys())

  return ranked[:top_k]

In [ ]:
# Define non-pet names
NOT_PET_NAMES = {
    "canine", "feline", "dog", "cat", "puppy", "kitten", "animal", "pet",
    "demodicosis", "dermatitis", "ringworm", "dermatophytosis",
    "infection", "disease", "condition", "syndrome",
    "treatment", "therapy", "diagnosis", "symptom", "clinical",
    "demodex", "microsporum", "trichophyton",
}

In [ ]:
# Pet name extractor
def extract_pet_name(query: str) -> str | None:
    entities = ner_model(query)

    for ent in entities:
        # Only consider PERSON entities with reasonable confidence
        if ent["entity_group"] == "PER" and ent["score"] >= 0.85:
            name = ent["word"].strip()

            # Blocklist check
            if name.lower() in NOT_PET_NAMES:
                continue

            # Reject single-char or numeric artifacts
            if len(name) < 2:
                continue

            return name.title()

    return None

In [ ]:
# Define example text profiles for each intent category
intent_profiles = {
  "database": """
  pet owner contact phone email address
  visit appointment history last visit when did
  medical record diagnosis prescribed medication treatment history
  who owns which vet pet profile weight vaccination
  """,

  "clinical": """
  what is how is disease condition symptoms clinical signs
  cause pathogenesis treatment therapy management
  ringworm demodicosis dermatitis fungal infection mite allergy
  how to treat how to diagnose veterinary medicine
  """,
}

# Convert each intent profile into an embedding vector
intent_embeddings = {
  intent: np.array(router_embedder.embed_query(text))
  for intent, text in intent_profiles.items()
}

In [ ]:
# Function to classify whether a query should use:
# - Clinical knowledge retrieval
# - Database retrieval
# - Or both
def classify_intent(
    query: str,
    clinical_threshold: float = 0.60,
    database_threshold: float = 0.50
) -> dict:
  """
  Database intent requires BOTH:
    1. A pet name detected in the query
    2. Database score >= database_threshold

  Clinical intent requires:
    1. Clinical score >= clinical_threshold
    OR fallback if nothing passes
  """
  # Convert the user query into an embedding vector
  q_emb = np.array(router_embedder.embed_query(query))

  # Compute similarity scores between the query embedding
  # and each intent embedding profile
  scores = {
      # Calculate dot-product similarity for each intent
      intent: np.dot(q_emb, np.array(emb))
      for intent, emb in intent_embeddings.items()
  }

  # Extract the pet name from the query using NER extraction
  pet_name = extract_pet_name(query)

  # Database retrieval only activates if:
  # 1. A pet name exists
  # 2. Database similarity score passes threshold
  use_database = pet_name is not None and scores["database"] >= database_threshold

  # Clinical retrieval activates if clinical similarity score passes threshold
  use_clinical = scores["clinical"] >= clinical_threshold

  return {
    "use_clinical": use_clinical,
    "use_database": use_database,
    "pet_name": pet_name,
    "scores": scores
  }

# **Indexing**

## Chunking

Breaks large documents into chunks before query into databases.

In [ ]:
# Clean text
def clean_text(text):
  text = text.replace("-\n", "")   # fix hyphen line breaks
  text = text.replace("\n", " ")   # flatten newlines
  text = re.sub(r"\s+", " ", text) # normalize spaces
  return text.strip()

In [ ]:
from nltk.tokenize import sent_tokenize
import nltk
import re

# punkt = sentence tokenizer model
# punkt_tab = dependency required in some newer NLTK builds
nltk.download("punkt")
nltk.download('punkt_tab')

# Split a long document into smaller, semantically coherent chunks by grouping sentences together up to a maximum character limit.
def sentence_chunker(text, max_chars=900):

  # Split input text into individual sentences
  sentences = sent_tokenize(text)

  # Define variables
  chunks = []

  # Temporary buffer for building a chunk
  current = ""

  for sent in sentences:

    # If adding this sentence stays within limit, keep appending
    if len(current) + len(sent) <= max_chars:
      current += " " + sent
    else:

      # If current chunk is not empty, finalize it
      if current.strip():
        chunks.append(current.strip())

      # Start new chunk with current sentence
      current = sent

  # Add final chunk if any remains
  if current.strip():
    chunks.append(current.strip())

  return chunks

In [ ]:
# Dectect sections
def detect_section(text):
  t = text.lower()

  # Detect references seciton
  if "references" in t or "bibliography" in t:
      return "references"

  # Detect clinical signs section
  if "clinical signs" in t or "clinical presentation" in t:
      return "clinical_signs"

  # Detect diagnosis-related section
  if re.search(r"\bdiagnos\w*\b", t):
      return "diagnosis"

  # Detect treatment-related section
  if "treat" in t or "therapy" in t or "management" in t:
      return "treatment"

  # Detect etiology or disease cause section
  if "etiolog" in t or "cause" in t or "pathogenesis" in t:
      return "etiology"

  return "other"

In [ ]:
# Filter noises
import re

# Detect noisy or low-value chunks
def is_noise(text):

  t = text.lower()

  # List of unwanted academic/reference patterns
  patterns = [
      r"doi:",
      r"received\s+\d+",
      r"accepted\s+\d+",
      r"copyright",
      r"references",
      r"bibliography",
      r"publisher’s note",
      r"disclaimer",
      r"conflict of interest",
      r"competing interests",
      r"author details",
      r"abbreviations",
  ]

  # Return True if any unwanted pattern is found
  if any(re.search(p, t) for p in patterns):
    return True

  return False

In [ ]:
def is_low_information(text):
  t = text.lower()

  # Too many capital names / affiliations = likely front matter
  if len(text.split()) < 30:
      return True

  # Too many commas = often author/address blocks
  if text.count(",") > 10:
      return True

  # Too many capitalized words (heuristic for author lists)
  caps = sum(1 for w in text.split() if w.istitle())
  if caps > 15:
      return True

  return False

In [ ]:
# Validate chunk size
def is_valid_chunk(text):

  # Keep even small chunks if they contain data or results
  has_numbers = any(char.isdigit() for char in text)
  has_citations = "(" in text and ")" in text

  # Remove very tiny garbage chunks
  if len(text) < 80:
    return False

  # Remove chunks with too few words
  if len(text.split()) < 25:
    return False

  # Never remove if it contains scientific evidence
  if has_numbers and len(text) > 50:
    return True

  return True

In [ ]:
# Create chunks
"""
Pipeline stages:
  1. Clean raw text
  2. Split into paragraphs
  3. Sentence-based chunking
  4. Filter low-quality / noisy content
  5. Attach structured metadata
"""
chunks = []

for doc in docs:

  # Raw document text from loader
  raw_text = doc.page_content

  # Clean text
  cleaned_text = clean_text(raw_text)

  # paragraph split
  paragraphs = cleaned_text.split("\n\n")

  for para in paragraphs:

    para = para.strip()

    # Skip empty paragraphs
    if not para:
        continue

    # sentence-based chunking
    split_chunks = sentence_chunker(para)

    for text in split_chunks:

      text = text.strip()

      # Skip empty chunks
      if not text:
          continue

      # detect section BEFORE filtering
      section = detect_section(text)

      # Skip reference/bibliography sections
      if section == "references":
          continue

      # Filtering
      if not is_valid_chunk(text):
          continue

      if is_noise(text):
          continue

      if is_low_information(text):
          continue

      # create a new document object for the chunk
      d = doc.model_copy() if hasattr(doc, "model_copy") else doc.copy()

      # Replace document content with chunk text
      d.page_content = text

      # Copy original metadata
      base_metadata = doc.metadata
      d.metadata = base_metadata.copy()

      # Enrich metadata for retrieval filtering
      d.metadata.update({
          "section": section,
          "species": base_metadata.get("species", "unknown"),
      })

      chunks.append(d)

print("Total chunks:", len(chunks))

## Embedding

Embeds text into vector and store into vectorstore.

In [ ]:
# Storing documents
vector_store.add_documents(chunks)

# **Retrieval**

## In-Memory (Semantic)

Vector and keyword fallback logic.

In [ ]:
# Group and retrieve chunks by disease category
from collections import defaultdict

# Initialize
disease_index = defaultdict(list)

for c in chunks:
  # Extract the disease label from chunk metadata
  disease = c.metadata.get("disease")

  if disease:
    disease_index[disease].append(c)

In [ ]:
from langchain_community.retrievers import BM25Retriever

def retrieve(query, k=5, top_diseases=1):
  # Route
  diseases = route_diseases(query, top_k=top_diseases)

  # Strict candidate pool
  candidate_chunks = []
  for d in diseases:
    candidate_chunks.extend(disease_index.get(d, []))

  if len(candidate_chunks) == 0:
    return [], diseases

  # Dual retrievers on same corpus
  bm25 = BM25Retriever.from_documents(candidate_chunks)
  bm25.k = k * 2

  bm25_docs = bm25.invoke(query)

  # Vector retriever (filtered implicitly via corpus)
  vector_docs = vector_store.as_retriever(
    search_kwargs={"k": k * 2}
  ).invoke(query)

  # Filter vector results to only routed diseases
  allowed_diseases = set(diseases)
  vector_docs = [
    d for d in vector_docs
    if d.metadata.get("disease") in allowed_diseases
  ]

  # Merge and deduplicate
  seen = set()
  merged = []

  for d in bm25_docs + vector_docs:
    key = (d.metadata.get("doc_id"), d.metadata.get("page"))
    if key not in seen:
        seen.add(key)
        merged.append(d)

  return merged[:k], diseases

In [ ]:
# Format retrieved document chunks into a readable string
def format_docs(docs):
  formatted = []
  references = []
  seen_refs = {}  # doc_id -> reference number
  ref_counter = 1

  for i, d in enumerate(docs, start=1):
    md = d.metadata

    # Build a unique key per source document (not per chunk)
    doc_id = md.get("doc_id", f"unknown_{i}")

    # Assign a reference number if not seen yet
    if doc_id not in seen_refs:
      seen_refs[doc_id] = ref_counter
      references.append({
        "num": ref_counter,
        "doc_id": doc_id,
        "title": md.get("title", "Unknown Title"),
        "journal": md.get("source_journal", "Unknown Journal"),
        "year": md.get("year", "n.d."),
        "url": md.get("url", ""),
      })
      ref_counter += 1

    ref_num = seen_refs[doc_id]

    text = f"""[DOCUMENT {i}] [Ref {ref_num}]

DOC_ID: {doc_id}
FILE: {md.get("file_name", "unknown")}
PAGE: {md.get("page", "unknown")}
SECTION: {md.get("section", "unknown")}
DISEASE: {md.get("disease", "unknown")}
SPECIES: {md.get("species", "unknown")}

CONTENT:
{d.page_content}"""

    formatted.append(text)

  # Build reference list string
  ref_list = "\n\nREFERENCES:\n"
  for r in references:
    ref_list += f"[{r['num']}] {r['title']}. {r['journal']} ({r['year']}). {r['url']}\n"

  return "\n\n---\n\n".join(formatted) + ref_list, references

## PostgreSQL (Structured)

In [ ]:
import pandas as pd
from sqlalchemy import text

# Execute a SQL query and return results as a DataFrame (Table)
def query_db(sql: str, params: dict = {}) -> pd.DataFrame:

  # Open a database connection
  with engine.connect() as conn:

    # Execute the SQL query with parameters
    result = conn.execute(text(sql), params)

    # Convert query result into a Pandas DataFrame
    return pd.DataFrame(result.fetchall(), columns=result.keys())

In [ ]:
# Get pet and owner information by pet name
def get_pet_info(pet_name: str) -> str:
  df = query_db("""
    SELECT p.name AS pet_name, p.species, p.breed,
           EXTRACT(YEAR FROM AGE(CURRENT_DATE, p.date_of_birth)) AS age,
           p.gender,
           o.full_name AS owner_name, o.phone, o.email
    FROM pets p
    JOIN owners o ON p.owner_id = o.owner_id
    WHERE LOWER(p.name) = LOWER(:name)
  """, {"name": pet_name})

  if df.empty:
    return f"No pet named '{pet_name}' found in the database."
  return df.to_string(index=False)

# Get visit history for a pet
def get_visit_history(pet_name: str) -> str:
  df = query_db("""
      SELECT v.visit_date, v.reason, v.weight_kg, v.temperature_c, v.notes
      FROM visits v
      JOIN pets p ON v.pet_id = p.pet_id
      WHERE LOWER(p.name) = LOWER(:name)
      ORDER BY v.visit_date DESC
      LIMIT 10
  """, {"name": pet_name})

  if df.empty:
    return f"No visit history found for '{pet_name}'."
  return df.to_string(index=False)

# Get medical records for pet
def get_medical_records(pet_name: str) -> str:
  df = query_db("""
      SELECT  mr.record_type, mr.title, mr.description,
              mr.data->>'severity' AS severity,
              mr.data->>'condition' AS condition
      FROM medical_records mr
      JOIN visits v ON mr.visit_id = v.visit_id
      JOIN pets p ON v.pet_id = p.pet_id
      WHERE LOWER(p.name) = LOWER(:name)
      ORDER BY v.visit_date DESC
      LIMIT 10
  """, {"name": pet_name})

  if df.empty:
    return f"No medical records found for '{pet_name}'."
  return df.to_string(index=False)

In [ ]:
# Sub-intent profiles for database table routing
db_intent_profiles = {
  "owner": "owner contact phone email address who owns",
  "visit": "visit appointment history last visit when did how many times",
  "medical": "medical record diagnosis prescribed medication treatment plan",
}

# Convert each sub-intent profile into embedding vectors
db_intent_embeddings = {
  k: np.array(router_embedder.embed_query(v))
  for k, v in db_intent_profiles.items()
}

# Determine which database table(s) should be queried
# based on semantic similarity between user query and intent profiles
def classify_db_intent(query: str, margin: float = 0.04) -> list[str]:
  q_emb = np.array(router_embedder.embed_query(query))

  scores = {
    k: np.dot(q_emb, emb)
    for k, emb in db_intent_embeddings.items()
  }

  # Find the intent with the highest score
  best_score = max(scores.values())

  # Return all intents above threshold
  matched = [k for k, s in scores.items() if best_score - s <= margin]

  # Fallback: return all if nothing matched
  return matched if matched else list(db_intent_profiles.keys())

In [ ]:
# Test
def debug_intent(query: str):
  q_emb = np.array(router_embedder.embed_query(query))

  result = classify_intent(query)

  print(f"Query: {query}")
  print(f"\nIntent scores:")
  for intent, score in result["scores"].items():
      print(f"  {intent}: {score:.4f}")

  print(f"\nuse_clinical : {result['use_clinical']}")
  print(f"use_database : {result['use_database']}")

  if result["use_database"]:
      print("\nDB sub-intent scores:")
      for k, emb in db_intent_embeddings.items():
          score = np.dot(q_emb, np.array(emb))
          print(f"  {k}: {score:.4f}")
  print()

debug_intent("Give me Bella’s medical records")
# debug_intent("What is canine demodicosis and how is it treated?")
# debug_intent("Milo was diagnosed with demodicosis, what should I do?")
# debug_intent("Hello there")

# **Generation**

## Qwen2.5 Generation

In [ ]:
# Generate a database-based response for the user's query
def answer_from_db(query: str) -> str:

  # Extract pet name from the query
  pet_name = extract_pet_name(query)

  if not pet_name:
      return "I couldn't identify a pet name. Please include the pet's name."

  # Determine which database sub-intents are relevant
  db_intents = classify_db_intent(query)

  # Build only relevant sections
  return build_patient_summary(
      pet_name=pet_name,
      db_intents=db_intents
  )

In [ ]:
def build_patient_summary(
    pet_name: str,
    db_intents: list[str] | None = None,
    include_all: bool = False
) -> str:
  parts = [f"=== PATIENT RECORD: {pet_name} ==="]

  # Default fallback
  if db_intents is None:
      db_intents = []

  # Include everything for clinical enrichment
  if include_all:
      db_intents = ["owner", "visit", "medical"]

  # Owner / Pet info
  if "owner" in db_intents:
      pet_info = get_pet_info(pet_name)

      if pet_info:
          parts.append(
              f"""=== Pet & Owner Info ===
{pet_info}"""
          )

  # Visit history
  if "visit" in db_intents:
      visits = get_visit_history(pet_name)

      if visits:
          parts.append(
              f"""=== Visit History ===
{visits}"""
          )

  # Medical records
  if "medical" in db_intents:
      records = get_medical_records(pet_name)

      if records:
          parts.append(
              f"""=== Medical Records ===
{records}"""
          )

  return "\n\n".join(parts)

In [ ]:
# Define a structured prompt template for generating veterinary answers
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""You are a veterinary clinical assistant used by veterinarians.
Answer the question using ONLY the information in the context below.
Be concise and clinical.
If patient records are provided, tailor your answer to that specific patient (breed, age, history, past diagnoses).
If the answer is not in the context, respond with: "Not found in context."

=== PATIENT CONTEXT ===
{patient_context}

=== CLINICAL KNOWLEDGE ===
{context}

Question: {question}

Answer:""")

In [ ]:
# Retrieve and format context
def retrieve_for_chain(inputs: dict) -> dict:
  global _last_references
  query = inputs["query"]
  docs, diseases = retrieve(query)
  context, references = format_docs(docs)
  _last_references = references

  return context

In [ ]:
# Strip everything up to and including "Answer:" if the prompt leaks through
def clean_output(text: str) -> str:
  if "Answer:" in text:
    text = text.split("Answer:")[-1]

  return text.strip()

In [ ]:
# Build RAG pipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Define the RAG pipeline using LangChain LCEL
rag_chain = (
  {
      "context": RunnableLambda(retrieve_for_chain),
      "question": RunnablePassthrough(),
  }
  | prompt  # Feed the structured input into the prompt template
  | llm # Send the formatted prompt to the LLM for generation
  | StrOutputParser()  # Convert LLM output into a clean string
  | RunnableLambda(clean_output) # Post-process output to remove prompt leakage
)

# **Full RAG pipeline**

In [ ]:
# Helper function to query the RAG system
# Helper function to query the RAG system
def ask(query):
  global _last_references

  intent = classify_intent(query)

  # Guard clause for irrelevant queries
  if not intent["use_database"] and not intent["use_clinical"]:
    return "Please ask a pet or veterinary-related question."

  parts = []
  patient_context_str = ""

  # Database path
  if intent["use_database"]:
    pet_name = intent["pet_name"]

    if pet_name:

      db_intents = classify_db_intent(query)

      # Build full patient summary for DB display AND prompt injection
      patient_context_str = build_patient_summary(
        pet_name=pet_name,
        db_intents=db_intents
      )
      parts.append(patient_context_str)
    else:
      parts.append("I couldn't identify a pet name. Please include the pet's name.")

  # RAG path
  if intent["use_clinical"]:

    # Format patient context block for prompt (empty if no patient)
    if patient_context_str:
      patient_block = f"=== PATIENT RECORD ===\n{patient_context_str}\n"
    else:
      patient_block = ""

    # Build enriched chain with patient context injected
    enriched_chain = (
        {
          "context": RunnableLambda(lambda q: retrieve_for_chain({"query": q})),
          "question": RunnablePassthrough(),
          "patient_context": RunnableLambda(lambda _: patient_block),
        }
        | prompt
        | llm
        | StrOutputParser()
        | RunnableLambda(clean_output)
    )

    response = enriched_chain.invoke(query)

    # Append numbered reference list to the answer
    if _last_references:
      response += "\n---\nReferences:"

      for r in _last_references:
        line = f"\n[{r['num']}] {r['title']}. {r['journal']} ({r['year']})"

        if r['url']:
          line += f". {r['url']}"
        response += line

    parts.append(response)

  return "\n\n".join(parts)

In [ ]:
def debug_retrieval(query):
  docs, diseases = retrieve(query)

  print(f"\nROUTED DISEASES: {diseases}")
  print(f"Retrieved {len(docs)} chunks\n")

  print(f"\nQUERY INTENT: {classify_intent(query)}")

  for i, d in enumerate(docs, start=1):
    print("-" * 80)
    print(f"CHUNK {i}")

    print("\nMETADATA:")
    for k, v in d.metadata.items():
        print(f"{k}: {v}")

    print("\nCONTENT:")
    print(d.page_content[:1200])
    print("\n")

In [ ]:
query = "What is a canine ringworm and how is it treated?"

print("\nQUESTION:")
print(query)

print("\nANSWER:")
print(ask(query))